# 12 — Cloud Security & AWS Fundamentals

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — Cloud Security  
**Certifications:** AWS Certified Security – Specialty, CompTIA CySA+

---

## Objectives
- Understand the AWS Shared Responsibility Model
- Review core AWS security services
- Audit IAM policies for misconfigurations using Python
- Detect S3 bucket exposure risks
- Analyse CloudTrail logs for suspicious API activity
- Apply CIS AWS Foundations Benchmark checks
- Generate a cloud security posture report

> ☁️ **Note:** This notebook uses simulated AWS data and `boto3`-style structures.  
> For live AWS use: `pip install boto3` and configure credentials via `aws configure`.

## 1. Setup & Imports

In [ ]:
import json
import re
from datetime import datetime, timezone
from collections import Counter, defaultdict

# boto3 for live AWS calls (optional)
try:
    import boto3
    BOTO3_AVAILABLE = True
    print('boto3 available — live AWS calls enabled')
except ImportError:
    BOTO3_AVAILABLE = False
    print('boto3 not installed — using simulated AWS data')
    print('Install with: pip install boto3')

## 2. AWS Shared Responsibility Model

```
┌─────────────────────────────────────────────────────────┐
│               CUSTOMER RESPONSIBILITY                    │
│  Data, Applications, IAM, OS patching, Network config   │
│  Encryption, Firewall rules, Client-side config         │
├─────────────────────────────────────────────────────────┤
│                  AWS RESPONSIBILITY                      │
│  Physical hardware, Global infrastructure, Hypervisor   │
│  Managed service software (RDS engine, Lambda runtime)  │
└─────────────────────────────────────────────────────────┘
```

### Core AWS Security Services

| Service | Purpose |
|---------|--------|
| **IAM** | Identity & access management — users, roles, policies |
| **CloudTrail** | API audit logging — who did what, when, from where |
| **GuardDuty** | Threat detection — ML-based anomaly detection |
| **Security Hub** | Centralised security findings aggregator |
| **Config** | Resource configuration compliance monitoring |
| **KMS** | Key management for encryption at rest/transit |
| **WAF** | Web Application Firewall for CloudFront / ALB |
| **Shield** | DDoS protection (Standard free, Advanced paid) |
| **Macie** | S3 data classification and sensitive data discovery |
| **Inspector** | Automated vulnerability assessment for EC2 / containers |

## 3. IAM Policy Analyser

In [ ]:
# Simulated IAM policies — mirrors real AWS policy JSON structure
SAMPLE_IAM_POLICIES = [
    {
        'PolicyName': 'DevTeamPolicy',
        'AttachedTo' : 'dev-team-role',
        'Document'  : {
            'Version'  : '2012-10-17',
            'Statement': [
                {'Effect': 'Allow', 'Action': 's3:*',   'Resource': '*'},
                {'Effect': 'Allow', 'Action': 'ec2:Describe*', 'Resource': '*'}
            ]
        }
    },
    {
        'PolicyName': 'AdminOverride',
        'AttachedTo' : 'contractor-user',
        'Document'  : {
            'Version'  : '2012-10-17',
            'Statement': [
                {'Effect': 'Allow', 'Action': '*', 'Resource': '*'}  # Full admin!
            ]
        }
    },
    {
        'PolicyName': 'LambdaExecPolicy',
        'AttachedTo' : 'lambda-execution-role',
        'Document'  : {
            'Version'  : '2012-10-17',
            'Statement': [
                {'Effect': 'Allow', 'Action': ['logs:CreateLogGroup', 'logs:PutLogEvents'], 'Resource': '*'},
                {'Effect': 'Allow', 'Action': 'iam:PassRole', 'Resource': '*'}  # Privilege escalation risk
            ]
        }
    },
    {
        'PolicyName': 'ReadOnlyAudit',
        'AttachedTo' : 'audit-role',
        'Document'  : {
            'Version'  : '2012-10-17',
            'Statement': [
                {'Effect': 'Allow', 'Action': ['cloudtrail:Get*', 'cloudtrail:List*',
                                               'config:Get*',     'iam:Get*'], 'Resource': '*'}
            ]
        }
    }
]


def analyse_iam_policy(policy: dict) -> list:
    """Check an IAM policy for common misconfigurations."""
    findings = []
    name     = policy['PolicyName']

    for stmt in policy['Document']['Statement']:
        if stmt['Effect'] != 'Allow':
            continue
        actions  = stmt['Action'] if isinstance(stmt['Action'], list) else [stmt['Action']]
        resource = stmt.get('Resource', '')

        # Check wildcard admin
        if '*' in actions:
            findings.append({'severity': 'CRITICAL', 'policy': name,
                             'issue': 'Wildcard Action (*) grants full AWS admin access',
                             'recommendation': 'Replace with least-privilege specific actions'})

        # Check broad S3 wildcard
        if 's3:*' in actions and resource == '*':
            findings.append({'severity': 'HIGH', 'policy': name,
                             'issue': 's3:* on all resources — allows read/write/delete on all buckets',
                             'recommendation': 'Scope to specific bucket ARNs e.g. arn:aws:s3:::my-bucket/*'})

        # Check iam:PassRole (privilege escalation vector)
        if 'iam:PassRole' in actions and resource == '*':
            findings.append({'severity': 'HIGH', 'policy': name,
                             'issue': 'iam:PassRole on * allows privilege escalation via role passing',
                             'recommendation': 'Restrict iam:PassRole to specific role ARNs'})

    return findings


all_findings = []
for policy in SAMPLE_IAM_POLICIES:
    findings = analyse_iam_policy(policy)
    all_findings.extend(findings)
    status = f'{len(findings)} issue(s)' if findings else 'Clean'
    print(f"  {policy['PolicyName']:<20} attached to {policy['AttachedTo']:<25} → {status}")
    for f in findings:
        print(f"    [{f['severity']:8}] {f['issue']}")

print(f'\nTotal IAM findings: {len(all_findings)}')

## 4. S3 Bucket Security Audit

In [ ]:
# Simulated S3 bucket configurations
S3_BUCKETS = [
    {'Name': 'company-public-assets',   'Public': True,  'Versioning': True,  'Encryption': True,  'Logging': True,  'MFA_Delete': False},
    {'Name': 'employee-hr-records',     'Public': False, 'Versioning': True,  'Encryption': True,  'Logging': True,  'MFA_Delete': True},
    {'Name': 'dev-team-scratch',        'Public': True,  'Versioning': False, 'Encryption': False, 'Logging': False, 'MFA_Delete': False},
    {'Name': 'cloudtrail-logs-archive', 'Public': False, 'Versioning': True,  'Encryption': True,  'Logging': True,  'MFA_Delete': True},
    {'Name': 'analytics-raw-data',      'Public': False, 'Versioning': False, 'Encryption': False, 'Logging': False, 'MFA_Delete': False},
]


def audit_s3_bucket(bucket: dict) -> list:
    """Check S3 bucket configuration against security best practices."""
    issues = []
    name   = bucket['Name']

    if bucket['Public'] and 'hr' in name.lower():
        issues.append({'severity': 'CRITICAL', 'check': 'Public access on sensitive bucket', 'bucket': name})
    elif bucket['Public'] and 'scratch' in name.lower():
        issues.append({'severity': 'HIGH', 'check': 'Public access on dev/scratch bucket — check for sensitive data', 'bucket': name})

    if not bucket['Encryption']:
        issues.append({'severity': 'HIGH', 'check': 'Server-side encryption not enabled', 'bucket': name,
                       'fix': 'Enable SSE-S3 or SSE-KMS'})

    if not bucket['Versioning']:
        issues.append({'severity': 'MEDIUM', 'check': 'Versioning disabled — no protection against accidental deletion', 'bucket': name})

    if not bucket['Logging']:
        issues.append({'severity': 'MEDIUM', 'check': 'Access logging disabled — no audit trail for bucket access', 'bucket': name})

    return issues


s3_findings = []
print('=== S3 BUCKET SECURITY AUDIT ===\n')
for bucket in S3_BUCKETS:
    issues = audit_s3_bucket(bucket)
    s3_findings.extend(issues)
    icon   = '🔓' if bucket['Public'] else '🔒'
    enc    = '✅' if bucket['Encryption'] else '❌'
    status = f'{len(issues)} issue(s)' if issues else '✓ Clean'
    print(f"  {icon} {bucket['Name']:<30} Enc={enc}  → {status}")
    for i in issues:
        print(f"    [{i['severity']:8}] {i['check']}")

print(f'\nTotal S3 findings: {len(s3_findings)}')

## 5. CloudTrail Log Analysis

In [ ]:
# Simulated CloudTrail API events (mirrors real CloudTrail JSON structure)
CLOUDTRAIL_EVENTS = [
    {'eventTime': '2025-01-16T01:10:00Z', 'eventName': 'ConsoleLogin',           'userIdentity': {'userName': 'jsmith'},       'sourceIPAddress': '203.0.113.5',  'errorCode': None,        'requestParameters': {}},
    {'eventTime': '2025-01-16T01:15:00Z', 'eventName': 'GetSecretValue',          'userIdentity': {'userName': 'jsmith'},       'sourceIPAddress': '203.0.113.5',  'errorCode': None,        'requestParameters': {'secretId': 'prod/db/password'}},
    {'eventTime': '2025-01-16T01:20:00Z', 'eventName': 'CreateUser',              'userIdentity': {'userName': 'jsmith'},       'sourceIPAddress': '203.0.113.5',  'errorCode': None,        'requestParameters': {'userName': 'backdoor-admin'}},
    {'eventTime': '2025-01-16T01:22:00Z', 'eventName': 'AttachUserPolicy',        'userIdentity': {'userName': 'jsmith'},       'sourceIPAddress': '203.0.113.5',  'errorCode': None,        'requestParameters': {'userName': 'backdoor-admin', 'policyArn': 'arn:aws:iam::aws:policy/AdministratorAccess'}},
    {'eventTime': '2025-01-16T01:30:00Z', 'eventName': 'PutBucketPolicy',         'userIdentity': {'userName': 'jsmith'},       'sourceIPAddress': '203.0.113.5',  'errorCode': None,        'requestParameters': {'bucketName': 'dev-team-scratch'}},
    {'eventTime': '2025-01-16T01:35:00Z', 'eventName': 'StopLogging',             'userIdentity': {'userName': 'jsmith'},       'sourceIPAddress': '203.0.113.5',  'errorCode': None,        'requestParameters': {'name': 'main-trail'}},
    {'eventTime': '2025-01-16T01:40:00Z', 'eventName': 'DeleteTrail',             'userIdentity': {'userName': 'jsmith'},       'sourceIPAddress': '203.0.113.5',  'errorCode': 'AccessDenied', 'requestParameters': {}},
    {'eventTime': '2025-01-16T02:00:00Z', 'eventName': 'DescribeInstances',       'userIdentity': {'userName': 'audit-role'},   'sourceIPAddress': '192.168.1.10', 'errorCode': None,        'requestParameters': {}},
    {'eventTime': '2025-01-16T02:05:00Z', 'eventName': 'AssumeRole',              'userIdentity': {'userName': 'lambda-exec'},  'sourceIPAddress': '10.0.0.5',     'errorCode': None,        'requestParameters': {'roleArn': 'arn:aws:iam::123456789:role/admin-role'}},
]

# High-risk API calls to alert on
SUSPICIOUS_APIS = {
    'CreateUser'          : ('CRITICAL', 'New IAM user created'),
    'AttachUserPolicy'    : ('CRITICAL', 'Policy attached to user — check for privilege escalation'),
    'CreateAccessKey'     : ('HIGH',     'New access key created'),
    'StopLogging'         : ('CRITICAL', 'CloudTrail logging disabled — defence evasion'),
    'DeleteTrail'         : ('CRITICAL', 'CloudTrail trail deletion attempted'),
    'PutBucketPolicy'     : ('HIGH',     'S3 bucket policy modified'),
    'GetSecretValue'      : ('HIGH',     'Secret retrieved from Secrets Manager'),
    'AssumeRole'          : ('MEDIUM',   'Role assumed — verify legitimacy'),
}

ct_alerts = []
print('=== CLOUDTRAIL THREAT DETECTION ===\n')
for event in CLOUDTRAIL_EVENTS:
    api = event['eventName']
    if api in SUSPICIOUS_APIS:
        sev, msg = SUSPICIOUS_APIS[api]
        alert = {
            'severity'  : sev,
            'api'       : api,
            'user'      : event['userIdentity'].get('userName', 'unknown'),
            'src_ip'    : event['sourceIPAddress'],
            'time'      : event['eventTime'],
            'message'   : msg,
            'error'     : event.get('errorCode')
        }
        ct_alerts.append(alert)
        error_note = f' [FAILED: {alert["error"]}]' if alert['error'] else ''
        print(f"  [{sev:8}] {event['eventTime']}  {api:<25} by {alert['user']:<15} from {alert['src_ip']}{error_note}")
        print(f"    → {msg}")

print(f'\n{len(ct_alerts)} suspicious API events detected')

## 6. CIS AWS Foundations Benchmark Checks

In [ ]:
# Simulated AWS account configuration state
ACCOUNT_CONFIG = {
    'root_mfa_enabled'        : False,   # CIS 1.5
    'root_access_keys_exist'  : True,    # CIS 1.4
    'password_policy': {
        'min_length'           : 8,      # CIS 1.8 — should be 14+
        'require_uppercase'    : True,
        'require_lowercase'    : True,
        'require_numbers'      : True,
        'require_symbols'      : False,  # CIS 1.9
        'max_age_days'         : 90,
        'prevent_reuse'        : 5
    },
    'cloudtrail_all_regions'  : True,    # CIS 2.1
    'cloudtrail_log_validation': False,  # CIS 2.2
    's3_cloudtrail_encrypted' : True,    # CIS 2.3
    'vpc_flow_logs_enabled'   : False,   # CIS 2.9
    'default_sg_unrestricted' : True,    # CIS 4.3
}

CIS_CHECKS = [
    ('1.4',  'Root account has no active access keys',   lambda c: not c['root_access_keys_exist'],   'CRITICAL'),
    ('1.5',  'MFA enabled for root account',             lambda c: c['root_mfa_enabled'],              'CRITICAL'),
    ('1.8',  'IAM password minimum length >= 14',        lambda c: c['password_policy']['min_length'] >= 14, 'MEDIUM'),
    ('1.9',  'IAM password requires symbols',            lambda c: c['password_policy']['require_symbols'],  'MEDIUM'),
    ('2.1',  'CloudTrail enabled in all regions',        lambda c: c['cloudtrail_all_regions'],        'HIGH'),
    ('2.2',  'CloudTrail log file validation enabled',   lambda c: c['cloudtrail_log_validation'],     'MEDIUM'),
    ('2.9',  'VPC flow logs enabled',                    lambda c: c['vpc_flow_logs_enabled'],         'MEDIUM'),
    ('4.3',  'Default security group restricts all traffic', lambda c: not c['default_sg_unrestricted'], 'HIGH'),
]

cis_results = []
print('=== CIS AWS FOUNDATIONS BENCHMARK ===\n')
print(f'{"CIS":<6} {"Status":<8} {"Severity":<10} Check')
print('-' * 75)
for cis_id, description, check_fn, severity in CIS_CHECKS:
    passed = check_fn(ACCOUNT_CONFIG)
    status = 'PASS' if passed else 'FAIL'
    icon   = '✅' if passed else '❌'
    print(f'{cis_id:<6} {icon} {status:<6} {severity:<10} {description}')
    if not passed:
        cis_results.append({'cis': cis_id, 'severity': severity, 'check': description, 'status': 'FAIL'})

passed_count = len(CIS_CHECKS) - len(cis_results)
print(f'\nResult: {passed_count}/{len(CIS_CHECKS)} checks passed  |  {len(cis_results)} failed')

## 7. Cloud Security Posture Report

In [ ]:
all_cloud_findings = all_findings + s3_findings + ct_alerts + cis_results

cspm_report = {
    'report_generated'  : datetime.now(timezone.utc).isoformat(),
    'account_id'        : '123456789012',
    'investigator'      : 'Bency (Benjamin Adjei)',
    'summary': {
        'iam_findings'       : len(all_findings),
        's3_findings'        : len(s3_findings),
        'cloudtrail_alerts'  : len(ct_alerts),
        'cis_failures'       : len(cis_results),
        'total_findings'     : len(all_cloud_findings),
        'severity_breakdown' : dict(Counter(
            f.get('severity', 'UNKNOWN') for f in all_cloud_findings
        ))
    },
    'top_priorities': [
        {'priority': 1, 'action': 'Enable MFA on root account immediately (CIS 1.5)'},
        {'priority': 2, 'action': 'Delete root account access keys (CIS 1.4)'},
        {'priority': 3, 'action': 'Investigate jsmith — backdoor-admin user created and CloudTrail stop attempted'},
        {'priority': 4, 'action': 'Remove wildcard Action (*) from AdminOverride policy on contractor-user'},
        {'priority': 5, 'action': 'Enable encryption and disable public access on dev-team-scratch bucket'},
        {'priority': 6, 'action': 'Enable VPC flow logs and CloudTrail log file validation'},
    ]
}

print(json.dumps(cspm_report, indent=2))

## 8. Key Takeaways

| Concept | Takeaway |
|---------|----------|
| **Shared Responsibility** | AWS secures the cloud; you secure what's *in* the cloud |
| **IAM least privilege** | Start with no permissions — add only what is needed |
| **Root account** | Never use root for day-to-day tasks; enable MFA, delete access keys |
| **CloudTrail** | Enable in all regions with log validation — it's your forensic record |
| **S3 public access** | Default Block Public Access should be ON for all new buckets |
| **CIS Benchmark** | Use as a baseline — automate checks with AWS Config Rules or Security Hub |
| **GuardDuty** | Enable it — $3/month for most accounts; detects what CloudTrail alone can't |

### AWS Security Checklist (Quick Reference)
```
✅ Enable MFA on root and all IAM users
✅ Delete root access keys
✅ Enable CloudTrail in all regions
✅ Enable GuardDuty
✅ Enable AWS Config
✅ Block S3 public access at account level
✅ Encrypt S3 buckets with KMS
✅ Enable VPC Flow Logs
✅ Restrict default security groups
✅ Use IAM roles for EC2 — never embed access keys in code
```

---
*Next: 13 — GRC: Governance, Risk & Compliance Frameworks*